# 4.2 The Prompt Pipeline: Engineering the Input

Interactive notebook for the **How AI Works** course. Run each cell top to bottom (Shift+Enter). The text and explanations live in the lesson page.

In [ ]:
# Setup: install the one dependency this course uses.
%pip install numpy --quiet

In [ ]:
# Our system prompt (the instructions)
SYSTEM_PROMPT = "You are a friendly math tutor. Explain concepts simply."

# A tiny database of context information
CONTEXT_DB = {
    "vector": "A vector is an ordered list of numbers like [1, 2, 3].",
    "dot product": "A dot product is the sum of multiplied corresponding elements."
}

def prompt_pipeline(user_query):
    # 1. Look for keywords in the query to find relevant context
    context_info = ""
    for keyword, info in CONTEXT_DB.items():
        if keyword in user_query.lower():
            context_info += f"\nContext: {info}"
    
    # 2. Template everything together
    final_prompt = f"""[SYSTEM]
{SYSTEM_PROMPT}

[CONTEXT]{context_info}

[USER]
{user_query}
"""
    return final_prompt

# Test our pipeline!
query = "What is a vector?"
formatted_prompt = prompt_pipeline(query)

print("The Final Prompt being sent to the LLM:")
print("-" * 30)
print(formatted_prompt)

## Exercises

<details>
<summary>Show solution</summary>

**Exercise 1:** Build a simple prompt pipeline that adds multiple contexts and formats them. Given a user query about "vectors," extract relevant keywords and add corresponding context.

```python
CONTEXT_DB = {
    "vector": "An ordered list of numbers representing a point in space.",
    "dimension": "The number of elements in a vector.",
    "magnitude": "The length or norm of a vector."
}

def extract_context(query, db):
    contexts = []
    for keyword, info in db.items():
        if keyword in query.lower():
            contexts.append(f"{keyword.capitalize()}: {info}")
    return "\n".join(contexts)

query = "What is the magnitude of a vector?"
context = extract_context(query, CONTEXT_DB)

final_prompt = f"""[SYSTEM]
You are a helpful math tutor.

[CONTEXT]
{context}

[USER]
{query}"""

print(final_prompt)
```

Expected output:
```text
[SYSTEM]
You are a helpful math tutor.

[CONTEXT]
Vector: An ordered list of numbers representing a point in space.
Magnitude: The length or norm of a vector.

[USER]
What is the magnitude of a vector?
```

</details>

<details>
<summary>Show solution</summary>

**Exercise 2:** Demonstrate the performance stage of the pipeline: truncate a long conversation to fit within a token budget.

```python
def approximate_tokens(text):
    return len(text) // 4

def truncate_history(history_text, max_tokens):
    tokens = approximate_tokens(history_text)
    if tokens <= max_tokens:
        return history_text, tokens
    
    # Keep only the most recent part
    max_chars = max_tokens * 4
    truncated = history_text[-max_chars:]
    return truncated, approximate_tokens(truncated)

long_history = "Turn 1: Hello\n" * 50
budget = 100

truncated, tokens = truncate_history(long_history, budget)
print(f"Original history tokens: ~{approximate_tokens(long_history)}")
print(f"Truncated history tokens: {tokens}")
print(f"Budget: {budget} tokens")
print(f"Fits in budget: {tokens <= budget}")
```

Expected output shows tokens reduced from ~200 to ~100.

</details>

<details>
<summary>Show solution</summary>

**Exercise 3:** Create a pipeline function that applies all stages: transform input, add context, template it, and return the final [prompt](../glossary.md#prompt).

```python
def full_pipeline(user_input, system_prompt, context_db):
    # Stage 1: Transform (lowercase, strip whitespace)
    cleaned_input = user_input.strip().lower()
    
    # Stage 2: Context Amplification
    context = ""
    for keyword, info in context_db.items():
        if keyword in cleaned_input:
            context += f"\n{info}"
    
    # Stage 3: Template
    final_prompt = f"""[SYSTEM]
{system_prompt}
[CONTEXT]{context}
[USER]
{user_input}"""
    
    # Stage 4: Could tokenize here (return as-is for now)
    return final_prompt

# Test it
db = {"vector": "A vector is an ordered list of numbers."}
result = full_pipeline("What is a vector?", "You are helpful.", db)
print(result)
```

This demonstrates how all pipeline stages work together to produce a [prompt](../glossary.md#prompt) ready for the model.

</details>

---

**Up Next:** What happens when our [context window](../glossary.md#context-window) isn't big enough? How can we give an AI access to millions of documents? Let's explore **Module 5: Expanding the AI's Brain (RAG & Tools)**.

*Try each exercise in a new cell below before opening the solution.*